In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *


# Append Mode

In [0]:
# Enabling Schema Inference 
spark.conf.set("spark.sql.streaming.streaming.schemaInference" ,"True")

In [0]:
df_batch = spark.read.format("json")\
        .option("multiLine",True)\
        .option("inferSchema",True)\
        .load("/Volumes/sparkcatalog/raw/source/src_append/")

json_schema = df_batch.schema

In [0]:
df = spark.readStream.format("json")\
        .option("multiLine",True)\
        .schema(json_schema)\
        .option("cleanSource","archive")\
        .option("sourceArchiveDir","/Volumes/sparkcatalog/raw/source/source_archive/")\
        .load("/Volumes/sparkcatalog/raw/source/src_append/")


In [0]:
query = df.writeStream.format("delta")\
        .outputMode("append")\
        .trigger(processingTime="5 seconds")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_append/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_append/data")\
        .start()

In [0]:
query.stop()

In [0]:
df_res = spark.read.format("delta")\
            .load("/Volumes/sparkcatalog/raw/source/sink_append/data")

display(df_res)

customer,items,metadata,order_id,payment,timestamp
"List(List(Vancouver, Canada, V5K 0A1), 502, alice@example.com, Alice Smith)","List(List(I102, 45.0, Bluetooth Keyboard, 1))","List(List(campaign, cyber_monday), List(channel, affiliate))",ORD1002,"List(PayPal, TXN7891)",2025-06-01T10:30:00Z
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))","List(List(campaign, back_to_school), List(channel, email))",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z


# Complete Mode

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sparkcatalog.raw.complete_source
(
  color STRING
)
USING DELTA

In [0]:
%sql
INSERT INTO sparkcatalog.raw.complete_source
VALUES ('red')

num_affected_rows,num_inserted_rows
1,1


In [0]:
df = spark.readStream.table('sparkcatalog.raw.complete_source')

df = df.groupBy('color').agg(count('color').alias('count'))

df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(processingTime="5 seconds")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()


In [0]:
df_sink = spark.read.format("delta")\
            .load("/Volumes/sparkcatalog/raw/source/sink_complete/data")

display(df_sink)

color,count
green,1
blue,1
red,2


# Trigger Modes

In [0]:
# Processing Time
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(processingTime="5 seconds")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()
    
# Once 
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(once=True)\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()

# Available Now
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(availableNow=True)\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()

# Continuous (Not Supported Yet With Delta Tables)
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(continuous='1 second')\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()

# ForEachBatch

In [0]:
df = spark.readStream.table('sparkcatalog.raw.complete_source')



In [0]:
def my_function(df,batch_id):

    # Writing To Stream 1 (Will Have To Use Batch Code)
    df.write.format("parquet")\
            .mode("overwrite")\
            .option("path","/Volumes/sparkcatalog/raw/source/streams/stream1")\
            .save()

    # Writing To Stream 2 (Will Have To Use Batch Code)
    df.write.format("parquet")\
            .mode("overwrite")\
            .option("path","/Volumes/sparkcatalog/raw/source/streams/stream2")\
            .save()

    # Can Use Batch Logic To Apply UPSERT 


In [0]:
df.writeStream.foreachBatch(my_function)\
        .outputMode("append")\
        .option("path","/Volumes/sparkcatalog/raw/source/streams/checkpoint")\
        .option("path","/Volumes/sparkcatalog/raw/source/streams/data")\
        .start()
        
        

# WINDOWS

In [0]:
%sql 
-- Source Table

CREATE TABLE sparkcatalog.raw.stream_source
(
  color STRING,
  event_time TIMESTAMP
)

In [0]:
%sql
select current_timestamp()

current_timestamp()
2026-02-12T18:34:18.36099Z


In [0]:
%sql
INSERT INTO sparkcatalog.raw.stream_source
VALUES
('GREEN','2026-02-12T20:08:18.360+00:00')

num_affected_rows,num_inserted_rows
1,1


In [0]:
df_str = spark.readStream.table('sparkcatalog.raw.stream_source')\
                .groupBy('color',window('event_time','10 minutes')).agg(count('color').alias("count"))

In [0]:
df_str.writeStream.format("delta")\
        .outputMode("complete")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/window_sink/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/window_sink/data")\
        .start()

In [0]:
df_res = spark.read.format("delta").load("/Volumes/sparkcatalog/raw/source/window_sink/data")
display(df_res)

color,window,count
GREEN,"List(2026-02-12T20:10:00Z, 2026-02-12T20:20:00Z)",1
GREEN,"List(2026-02-12T20:00:00Z, 2026-02-12T20:10:00Z)",2
BLUE,"List(2026-02-12T20:10:00Z, 2026-02-12T20:20:00Z)",1
RED,"List(2026-02-12T20:00:00Z, 2026-02-12T20:10:00Z)",2


### WATERMARKS

In [0]:
df_str = spark.readStream.table('sparkcatalog.raw.stream_source')\
                .withWatermark('event_time','10 minutes')\
                .groupBy('color',window('event_time','10 minutes')).agg(count('color').alias("count"))